# LOB Lookup

This notebook reads the SQLite database produced by `scripts.poll_orderbooks` and walks through three layers of analysis:

1. `token_id` level: one order book per outcome.
2. `condition_id` level: one market as a `Yes/No` pair.
3. `event` level: the full set of markets from one Polymarket event.

The intended workflow is simple: find the timestamp you care about, inspect the outcome-level book, move up to the market pair, and only then summarize the whole event.

## 1. Open The Database

By default, the notebook reads `cached_data/iran_conflict_orderbooks.sqlite`. Change `DB_PATH` below if your database lives somewhere else.

In [ ]:
from pathlib import Path
import sqlite3

from IPython.display import display
import pandas as pd

DB_PATH = Path("../db/iran_conflict_orderbooks.sqlite")
# DB_PATH = Path("../db/us_forces.sqlite")
PROCESSED_DIR = Path("../cached_cached_data/processed")

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

print(DB_PATH.resolve())


/Users/sneddy/research/polymarket_research/db/us_forces.sqlite


## 2. Helper Functions

The helpers below are intentionally small and explicit. They let you:
- inspect available polling cycles
- inspect available markets and outcomes
- select the last snapshot at or before `as_of_utc`
- load top-of-book, full depth, market-pair views, and an event summary
- export outputs to `csv` or `parquet` under `cached_data/processed`


In [15]:
def query_df(sql: str, params: tuple = ()) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as conn:
        return pd.read_sql_query(sql, conn, params=params)


def list_poll_cycles(limit: int = 20) -> pd.DataFrame:
    return query_df(
        """
        SELECT
            id AS poll_cycle_id,
            captured_at_utc,
            interval_seconds,
            levels_requested,
            outcomes_expected,
            outcomes_succeeded,
            outcomes_failed
        FROM poll_cycles
        ORDER BY captured_at_utc DESC
        LIMIT ?
        """,
        (limit,),
    )


def list_markets() -> pd.DataFrame:
    return query_df(
        """
        SELECT
            source_slug,
            market_slug,
            outcome_name,
            token_id,
            condition_id,
            market_question,
            group_item_title
        FROM market_outcomes
        ORDER BY market_slug, outcome_index
        """
    )


def _selected_cycle_cte(as_of_utc: str | None) -> tuple[str, tuple]:
    if as_of_utc is None:
        return (
            """
            WITH selected_cycle AS (
                SELECT id, captured_at_utc
                FROM poll_cycles
                ORDER BY captured_at_utc DESC
                LIMIT 1
            )
            """,
            (),
        )

    return (
        """
        WITH selected_cycle AS (
            SELECT id, captured_at_utc
            FROM poll_cycles
            WHERE captured_at_utc <= ?
            ORDER BY captured_at_utc DESC
            LIMIT 1
        )
        """,
        (as_of_utc,),
    )


def load_top_of_book(
    as_of_utc: str | None = None,
    market_slug: str | None = None,
    outcome_name: str | None = None,
) -> pd.DataFrame:
    cte_sql, cte_params = _selected_cycle_cte(as_of_utc)
    sql = cte_sql + """
    SELECT
        sc.captured_at_utc AS selected_captured_at_utc,
        s.market_slug,
        s.outcome_name,
        s.token_id,
        s.condition_id,
        s.best_bid,
        s.best_ask,
        CASE
            WHEN s.best_bid IS NOT NULL AND s.best_ask IS NOT NULL THEN (s.best_bid + s.best_ask) / 2.0
            ELSE NULL
        END AS mid_price,
        CASE
            WHEN s.best_bid IS NOT NULL AND s.best_ask IS NOT NULL THEN s.best_ask - s.best_bid
            ELSE NULL
        END AS spread,
        s.last_trade_price,
        s.tick_size,
        s.min_order_size,
        s.book_timestamp_ms,
        s.book_hash,
        s.bids_count,
        s.asks_count
    FROM selected_cycle sc
    JOIN orderbook_snapshots s ON s.poll_cycle_id = sc.id
    WHERE (? IS NULL OR s.market_slug = ?)
      AND (? IS NULL OR s.outcome_name = ?)
    ORDER BY s.market_slug, s.outcome_name
    """
    params = cte_params + (market_slug, market_slug, outcome_name, outcome_name)
    return query_df(sql, params)


def load_levels(
    as_of_utc: str | None = None,
    market_slug: str | None = None,
    outcome_name: str | None = None,
    side: str | None = None,
) -> pd.DataFrame:
    cte_sql, cte_params = _selected_cycle_cte(as_of_utc)
    sql = cte_sql + """
    SELECT
        sc.captured_at_utc AS selected_captured_at_utc,
        s.market_slug,
        s.outcome_name,
        s.token_id,
        s.condition_id,
        l.side,
        l.level_index,
        l.price,
        l.size,
        l.price * l.size AS notional
    FROM selected_cycle sc
    JOIN orderbook_snapshots s ON s.poll_cycle_id = sc.id
    JOIN orderbook_levels l ON l.snapshot_id = s.id
    WHERE (? IS NULL OR s.market_slug = ?)
      AND (? IS NULL OR s.outcome_name = ?)
      AND (? IS NULL OR l.side = ?)
    ORDER BY s.market_slug, s.outcome_name, l.side, l.level_index
    """
    params = cte_params + (market_slug, market_slug, outcome_name, outcome_name, side, side)
    return query_df(sql, params)


def load_market_pair_view(
    as_of_utc: str | None = None,
    market_slug: str | None = None,
) -> pd.DataFrame:
    top = load_top_of_book(as_of_utc=as_of_utc, market_slug=market_slug)
    if top.empty:
        return top

    pair = top.pivot_table(
        index=["selected_captured_at_utc", "market_slug", "condition_id"],
        columns="outcome_name",
        values=["best_bid", "best_ask", "mid_price", "spread", "last_trade_price"],
        aggfunc="first",
    )
    pair.columns = [f"{metric}_{outcome.lower()}" for metric, outcome in pair.columns]
    pair = pair.reset_index()

    for col in ["mid_price_yes", "mid_price_no", "best_bid_yes", "best_bid_no", "best_ask_yes", "best_ask_no"]:
        if col not in pair.columns:
            pair[col] = pd.NA

    pair["mid_sum_yes_no"] = pair["mid_price_yes"] + pair["mid_price_no"]
    pair["bid_sum_yes_no"] = pair["best_bid_yes"] + pair["best_bid_no"]
    pair["ask_sum_yes_no"] = pair["best_ask_yes"] + pair["best_ask_no"]
    pair["mid_dislocation_vs_1"] = pair["mid_sum_yes_no"] - 1.0
    return pair.sort_values(["market_slug"])


def load_event_summary(as_of_utc: str | None = None) -> pd.DataFrame:
    pair = load_market_pair_view(as_of_utc=as_of_utc)
    if pair.empty:
        return pair

    summary = (
        pair.groupby("selected_captured_at_utc", as_index=False)
        .agg(
            markets=("condition_id", "nunique"),
            mean_yes_mid=("mid_price_yes", "mean"),
            mean_no_mid=("mid_price_no", "mean"),
            mean_mid_sum_yes_no=("mid_sum_yes_no", "mean"),
            mean_mid_dislocation_vs_1=("mid_dislocation_vs_1", "mean"),
        )
    )
    return summary


def export_df(df: pd.DataFrame, path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.suffix == ".csv":
        df.to_csv(out, index=False)
    elif out.suffix == ".parquet":
        df.to_parquet(out, index=False)
    else:
        raise ValueError("Supported suffixes: .csv, .parquet")
    return out


## 3. Inspect What Was Collected

Start by checking which polling cycles and which markets are already in the database. In practice this is how most lookups begin: first find the timestamp, then narrow to the market and outcome you care about.

In [16]:
poll_cycles = list_poll_cycles(limit=20)
markets = list_markets()

display(poll_cycles)
display(markets)


## 4. Choose A Timestamp And A Slice

Selection logic:
- `AS_OF_UTC = None` means use the latest available poll cycle.
- If you need a historical slice, provide a UTC timestamp and the notebook will pick the last snapshot at or before that time.
- Leave `MARKET_SLUG` and `OUTCOME_NAME` as `None` if you want to inspect the whole event first.

In [23]:
AS_OF_UTC = None
MARKET_SLUG = None
# MARKET_SLUG = "iran-x-israelus-conflict-ends-by-march-31"
OUTCOME_NAME = None
SIDE = None  # None, 'bid', or 'ask'

top_of_book_df = load_top_of_book(
    as_of_utc=AS_OF_UTC,
    market_slug=MARKET_SLUG,
    outcome_name=OUTCOME_NAME,
)

levels_df = load_levels(
    as_of_utc=AS_OF_UTC,
    market_slug=MARKET_SLUG,
    outcome_name=OUTCOME_NAME,
    side=SIDE,
)

display(top_of_book_df)
display(levels_df.head(10))


## 5. Outcome-Level View: Analyze The Book As `token_id`

This is the base layer for order book analysis.

At this level, you typically care about:
- spread
- top-of-book
- depth by level
- bid/ask imbalance
- the shape of the book for one specific outcome

The next cell reshapes the levels into a wide format. `condition_id` is included in the index explicitly so different markets cannot be mixed together.

In [18]:
wide_levels_df = (
    levels_df.assign(level_key=lambda df: df["side"] + "_" + df["level_index"].astype(str))
    .pivot_table(
        index=["selected_captured_at_utc", "market_slug", "outcome_name", "token_id", "condition_id"],
        columns="level_key",
        values=["price", "size", "notional"],
        aggfunc="first",
    )
)

if not wide_levels_df.empty:
    wide_levels_df.columns = [f"{value}_{level}" for value, level in wide_levels_df.columns]
    wide_levels_df = wide_levels_df.reset_index().sort_values(["market_slug", "outcome_name"])

display(wide_levels_df)


## 6. Market-Level View: Join `Yes` And `No` By `condition_id`

Once the individual outcomes are clear, the next step is the market pair.

Useful checks at this level:
- `mid_price_yes + mid_price_no`
- `best_bid_yes + best_bid_no`
- `best_ask_yes + best_ask_no`
- deviation of the sum of mids from `1.0`

This is less about the microstructure of one book and more about whether the market is internally consistent.

In [24]:
market_pair_df = load_market_pair_view(as_of_utc=AS_OF_UTC, market_slug=MARKET_SLUG)
display(market_pair_df)


## 7. Event-Level View: Summarize The Full Series

At the event level, the goal is no longer one market. Instead, you want to understand the entire term structure of markets behind one event URL.

This is useful when you want to see:
- how implied probabilities are distributed across deadlines
- whether dislocation is systematic across the whole series
- how many markets are currently liquid and internally consistent

In [20]:
event_summary_df = load_event_summary(as_of_utc=AS_OF_UTC)
display(event_summary_df)


## 8. Export Processed Artifacts

In most workflows it is useful to persist three layers separately:
- `top_of_book_df` for a fast snapshot overview
- `wide_levels_df` for downstream feature engineering
- `market_pair_df` for market-level consistency checks

All outputs below are written to `cached_data/processed`.

In [10]:
EXPORT_TOP_OF_BOOK = PROCESSED_DIR / "lob_lookup_top_of_book.csv"
EXPORT_WIDE_LEVELS = PROCESSED_DIR / "lob_lookup_wide_levels.csv"
EXPORT_MARKET_PAIR = PROCESSED_DIR / "lob_lookup_market_pair.csv"

if not top_of_book_df.empty:
    print("top_of_book ->", export_df(top_of_book_df, EXPORT_TOP_OF_BOOK))

if not wide_levels_df.empty:
    print("wide_levels ->", export_df(wide_levels_df, EXPORT_WIDE_LEVELS))

if not market_pair_df.empty:
    print("market_pair ->", export_df(market_pair_df, EXPORT_MARKET_PAIR))


top_of_book -> ../cached_cached_data/processed/lob_lookup_top_of_book.csv
wide_levels -> ../cached_cached_data/processed/lob_lookup_wide_levels.csv
market_pair -> ../cached_cached_data/processed/lob_lookup_market_pair.csv
